# Notebook 01 — Stereo Depth (RAFT-Stereo)

Downloads SceneFlow, trains RAFT-Stereo on indoor data, exports ONNX for Jetson TensorRT.

## Cell 01-01 | Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch

BASE   = '/content/drive/MyDrive/ARGUS'
DS     = f'{BASE}/datasets'
MODELS = f'{BASE}/models'
CKPTS  = f'{BASE}/checkpoints'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | Drive: {os.path.exists(BASE)}')

## Cell 01-02 | Clone RAFT-Stereo Repository

In [ ]:
RAFT_DIR = f'{BASE}/raft_stereo'

if not os.path.exists(RAFT_DIR):
    os.system(f'git clone https://github.com/princeton-vl/RAFT-Stereo {RAFT_DIR}')
    print('RAFT-Stereo cloned to Drive')
else:
    print('RAFT-Stereo already exists')

os.chdir(RAFT_DIR)
os.system('pip install -q einops')
print('RAFT-Stereo directory:')
os.system('ls')

## Cell 01-03 | Download SceneFlow Dataset

In [ ]:
import subprocess, os

SF_DIR = f'{DS}/sceneflow'
os.makedirs(SF_DIR, exist_ok=True)

if os.path.exists(f'{SF_DIR}/flyingthings3d_done.flag'):
    print('SceneFlow already downloaded')
else:
    print('Downloading SceneFlow FlyingThings3D subset (~10GB)...')
    files = [
        ('https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/raw_data/FlyingThings3D_subset_image_clean.tar.gz', 'FlyingThings3D_image_clean.tar.gz'),
        ('https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/raw_data/FlyingThings3D_subset_disparity.tar.gz', 'FlyingThings3D_disparity.tar.gz'),
    ]
    for url, fname in files:
        out = f'{SF_DIR}/{fname}'
        if not os.path.exists(out):
            subprocess.run(['wget', '-q', '--show-progress', '-O', out, url])
            subprocess.run(['tar', '-xzf', out, '-C', SF_DIR])
    open(f'{SF_DIR}/flyingthings3d_done.flag', 'w').close()
    print('SceneFlow download complete')
os.system(f'du -sh {SF_DIR}')

## Cell 01-04 | Download Pretrained RAFT-Stereo Weights

In [ ]:
import urllib.request

WEIGHTS_DIR  = f'{MODELS}/depth'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
weights_url  = 'https://github.com/princeton-vl/RAFT-Stereo/releases/download/v0.0.0/raftstereo-realtime.pth'
weights_path = f'{WEIGHTS_DIR}/raftstereo-realtime.pth'

if not os.path.exists(weights_path):
    print('Downloading RAFT-Stereo pretrained weights...')
    urllib.request.urlretrieve(weights_url, weights_path)
    print(f'Weights saved to {weights_path}')
else:
    print(f'Weights already exist at {weights_path}')

size = os.path.getsize(weights_path) / 1e6
print(f'File size: {size:.1f} MB')

## Cell 01-05 | Custom Indoor Dataset Loader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

class StereoDataset(Dataset):
    # Custom stereo dataset for AR0234 indoor image pairs
    def __init__(self, root, split='train', img_size=(480, 640)):
        self.left_dir  = f'{root}/{split}/left'
        self.right_dir = f'{root}/{split}/right'
        self.disp_dir  = f'{root}/{split}/disp'
        self.img_size  = img_size
        self.samples   = sorted([f for f in os.listdir(self.left_dir) if f.endswith(('.png', '.jpg'))])
        print(f'  {split}: {len(self.samples)} stereo pairs found')
        self.transform = A.Compose([
            A.RandomCrop(img_size[0], img_size[1]),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.5),
            A.ToFloat(max_value=255),
            ToTensorV2(),
        ], additional_targets={'right': 'image'})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        name  = self.samples[idx]
        stem  = os.path.splitext(name)[0]
        left  = np.array(Image.open(f'{self.left_dir}/{name}').convert('RGB'))
        right = np.array(Image.open(f'{self.right_dir}/{name}').convert('RGB'))
        disp  = np.load(f'{self.disp_dir}/{stem}.npy').astype(np.float32)
        out   = self.transform(image=left, right=right)
        disp_t = torch.from_numpy(disp[:self.img_size[0], :self.img_size[1]]).unsqueeze(0)
        return {'left': out['image'], 'right': out['right'], 'disparity': disp_t}

CUSTOM_STEREO = f'{DS}/custom_stereo'
for d in [f'{CUSTOM_STEREO}/train/left', f'{CUSTOM_STEREO}/train/right', f'{CUSTOM_STEREO}/train/disp']:
    os.makedirs(d, exist_ok=True)
print('Custom stereo dataset structure ready.')
print('Add AR0234 stereo pairs before running the training cell.')

## Cell 01-06 | RAFT-Stereo Training Loop

In [ ]:
import sys
sys.path.insert(0, RAFT_DIR)
from core.raft_stereo import RAFTStereo
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn.functional as F

args = type('Args', (), {
    'hidden_dims': [128]*3, 'context_dims': [128]*3,
    'corr_implementation': 'reg', 'shared_backbone': False,
    'corr_levels': 4, 'corr_radius': 4, 'n_downsample': 2,
    'slow_fast_gru': True, 'n_gru_layers': 3, 'mixed_precision': True,
})()

model = RAFTStereo(args).to(DEVICE)
ckpt  = torch.load(f'{MODELS}/depth/raftstereo-realtime.pth', map_location=DEVICE)
model.load_state_dict(ckpt, strict=False)
print('Pretrained weights loaded')

CUSTOM_STEREO = f'{DS}/custom_stereo'
has_custom    = len(os.listdir(f'{CUSTOM_STEREO}/train/left')) > 0
dataset       = StereoDataset(CUSTOM_STEREO, 'train', img_size=(320, 480))
loader        = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = OneCycleLR(optimizer, max_lr=1e-4, steps_per_epoch=len(loader), epochs=20, pct_start=0.05)
scaler    = torch.cuda.amp.GradScaler()

EPOCHS    = 20
CKPT_PATH = f'{CKPTS}/raft_stereo_indoor'
os.makedirs(CKPT_PATH, exist_ok=True)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for i, batch in enumerate(loader):
        left = batch['left'].to(DEVICE)
        right = batch['right'].to(DEVICE)
        disp  = batch['disparity'].to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            preds = model(left, right, iters=12)
            loss  = 0.0
            for j, pred in enumerate(preds):
                w    = 0.9 ** (len(preds) - j - 1)
                mask = (disp > 0) & (disp < 192)
                loss += w * F.smooth_l1_loss(pred[mask], disp[mask])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        epoch_loss += loss.item()
        if i % 10 == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} | Step {i}/{len(loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch+1} avg loss: {epoch_loss/len(loader):.4f}')
    if (epoch + 1) % 5 == 0:
        torch.save({'epoch': epoch+1, 'model_state': model.state_dict()},
                   f'{CKPT_PATH}/epoch_{epoch+1}.pth')

torch.save(model.state_dict(), f'{MODELS}/depth/raft_stereo_final.pth')
print('Training complete. Final model saved to Drive.')

## Cell 01-07 | Export to ONNX

In [ ]:
model.eval()
H, W         = 480, 640
dummy_left   = torch.randn(1, 3, H, W).to(DEVICE)
dummy_right  = torch.randn(1, 3, H, W).to(DEVICE)
ONNX_PATH    = f'{EXPORTS}/tensorrt/raft_stereo_640x480.onnx'
os.makedirs(os.path.dirname(ONNX_PATH), exist_ok=True)

class RAFTStereoExport(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, l, r): return self.model(l, r, iters=12, test_mode=True)

with torch.no_grad():
    torch.onnx.export(
        RAFTStereoExport(model), (dummy_left, dummy_right), ONNX_PATH,
        input_names=['left', 'right'], output_names=['disparity'],
        dynamic_axes={'left': {0: 'batch'}, 'right': {0: 'batch'}, 'disparity': {0: 'batch'}},
        opset_version=17, do_constant_folding=True,
    )
print(f'ONNX exported: {ONNX_PATH}')
print(f'Size: {os.path.getsize(ONNX_PATH)/1e6:.1f} MB')
print('On Jetson: trtexec --onnx=raft_stereo_640x480.onnx --saveEngine=raft_stereo_fp16.trt --fp16')